<h1 style=\"text-align: center; font-size: 50px;\"> 📦 Register Model </h1>

This notebook packages the **audio-native agentic workflow** as an **MLflow pyfunc model**, logs it with artifacts
(index, config), and registers it to the MLflow Model Registry for serving.

- Retrieval: **CLAP** audio embeddings over timestamped windows (+ **MMR** reranker)
- Generation: **Qwen Omni** listens to the selected audio windows and answers (no transcripts required)
- Orchestration: **LangGraph** (relevance → memory → retrieve → rerank → answer → memoize)
- Vector store: **FAISS** (in-model artifact or built on first run)
- Memory: disk-backed key-value cache (per-corpus+question)


# Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Verify Assets
- KV Memory
- MLflow Registration
- Load Model and Test Payload
- Message History

# Start Execution

In [1]:
# Standard library imports
import os  # Provides OS-related utilities
import sys  # Allows manipulation of Python runtime environment
import time  # Enables time-based operations
from pathlib import Path  # Object-oriented file system paths

# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.utils import (  # Utility functions for logging, LLM I/O, and schema generation
    load_config,
    load_secrets,
    load_secrets_to_env,
    get_project_root,
    logger,
    setup_model_environment,
)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

# Install and Import Libraries

In [3]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 79.3 ms, sys: 60.3 ms, total: 140 ms
Wall time: 3.23 s


In [4]:
from __future__ import annotations  # Enables postponed evaluation of annotations (PEP 563)

# ─────── Standard Library ───────
import json  # JSON serialization and deserialization
import warnings  # Issue warning messages
from collections import namedtuple  # Factory for creating tuple subclasses with named fields
from datetime import datetime  # Date and time utilities
from pathlib import Path  # Object-oriented filesystem paths
from typing import Any, Dict, List, Literal, Optional, TypedDict  # Type hinting support
import numpy as np  # Numerical operations and array handling
import soundfile as sf  # Reading and writing sound files

# ─────── Third-Party Packages ───────
import mlflow  # Model tracking and serving framework
import mlflow.pyfunc  # MLflow Python function interface for custom models
from mlflow.tracking import MlflowClient  # Interface to interact with MLflow tracking server for experiments, runs, and artifacts
from IPython.display import Markdown, display  # IPython utilities for notebook output formatting
from tqdm import tqdm  # Visual progress bar for loops
import torch # PyTorch for tensor computations and deep learning
import torchaudio # Audio processing library built on PyTorch
import faiss # Library for efficient similarity search and clustering of dense vectors
import pandas as pd # Data manipulation and analysis

# Qwen Omni (audio+video+text) – both full & Thinker-only variants
from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration # Qwen Omni processor and model
from transformers import AutoProcessor as ClapProcessor, ClapModel # CLAP processor and model for audio embeddings
from qwen_omni_utils import process_mm_info     # official utils to prep audio/video inputs

# ─────── LangChain Core & Community ───────
from langgraph.graph import StateGraph, END

# ─────── Local application-specific imports ───────
from src.agentic_workflow import build_audio_agentic_graph # Custom workflow builder for agentic tasks
from src.model_selection import ModelSelector
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state
from src.simple_kv_memory import _mem_get, _mem_put  # Functions for getting and putting items in memory
from src.generate_test_audio import generate_test_audio, generate_and_convert_formats  # Functions to generate test audio files
from src.segment_audio_embeddings import (  # Functions for segmenting audio and extracting embeddings
    clap_embed_audio,
    clap_embed_text,
    segment_audio_embeddings, 
    rerank_hits_mmr, 
    retrieve_and_rerank,
    ensure_wav
)
from src.agentic_audio_rag_model import AudioAgenticPyFunc
from core.agentic_audio_rag_service.agentic_audio_rag_service import AgenticAudioService




/opt/conda/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


# Configure Settings

In [5]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [ ]:
project_root = get_project_root()

CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"
MEDIA_DIR: Path = Path("../data/input/meeting_recording")
DATA_PATH = "../data/input/meeting_recording"
DEMO_FOLDER = "../demo"
MEMORY_PATH: Path = Path("../data/memory")

MLFLOW_EXPERIMENT_NAME = "AIStudio-Agentic-Audio-RAG-Experiment"
MLFLOW_RUN_NAME = "AIStudio-Agentic-Audio-RAG-Run"
MLFLOW_MODEL_NAME = "AIStudio-Agentic-Audio-RAG-Model"

In [7]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

✅ Loaded 1 secrets into environment variables.
✅ Configuration loaded successfully
✅ Secrets loaded successfully


In [8]:
logger.info('Notebook execution started.')

## Verify Assets

In [9]:
def log_asset_status(asset_path: str, asset_name: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured.")
    else:
        logger.info(f"{asset_name} is not properly configured. Please ensure the required asset is correctly configured in your AI Studio project according to the README file.")

def log_secrets_status(secrets: Dict[str, Any], success_message: str, failure_message: str) -> None:
    """
    Logs the status of secrets based on their existence.

    Parameters:
        secrets (Dict[str, Any]): Secrets retrieved to check if they exist.
        success_message (str): Message to log if secrets exists.
        failure_message (str): Message to log if secrets do not exist.
    """
    if secrets:
        logger.info(f"Project secrets are available. {success_message}")
    else:
        logger.info(f"There are no project secrets found. {failure_message}")

In [10]:
log_asset_status(
    asset_path=MEDIA_DIR,
    asset_name="Input Data",
)

log_asset_status(
    asset_path=CONFIG_PATH,
    asset_name="Config",
)

log_secrets_status(
    secrets=secrets,
    success_message="",
    failure_message="Please check if the secrets were propely connfigured in your secrets yaml file or in Secrets Manager."
)

# KV Memory

In [11]:
memory: SimpleKVMemory = SimpleKVMemory(MEMORY_PATH)
memory.set('dummy key', 'dummy value')

# MLflow Registration

In [12]:
%%time

from packaging.version import parse as vparse

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {MLFLOW_EXPERIMENT_NAME}")

MEMORY_PATH: Path = Path("../data/memory")

# === Get model path from config ===
model_path = config.get("model_path")
if model_path and os.path.exists(model_path):
    logger.info(f"✅ Model file found at: {model_path}")
else:
    logger.info(f"⚠️ Warning: Model file not found at {model_path}. Please verify the path in config.yaml.")

logger.info(f'Starting the experiment: {MLFLOW_EXPERIMENT_NAME}')
logger.info(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")

with mlflow.start_run(run_name=f"register-{MLFLOW_RUN_NAME}") as run:
    # Print the artifact URI for reference
    logger.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    
     # Log model artifacts using custom ChatbotService
    AgenticAudioService.log_model(
        artifact_path=MLFLOW_MODEL_NAME,
        config_path=CONFIG_PATH,
        docs_path=DATA_PATH,
        secrets_dict=secrets if secrets else None,
        model_path=model_path,
        demo_folder=DEMO_FOLDER
    )
    
    # Construct the URI for the logged model
    model_uri = f"runs:/{run.info.run_id}/{MLFLOW_MODEL_NAME}"

    # Register the model into MLflow Model Registry
    mlflow.register_model(
        model_uri=model_uri,
        name=MLFLOW_MODEL_NAME
    )

print("Logged model at:", model_uri)
print("Registered name:", MLFLOW_MODEL_NAME)
logger.info(f"✅ Model registered successfully with run ID: {run.info.run_id}")

#######################################################################################


Using MLflow tracking URI: /phoenix/mlflow
Experiment: AIStudio-Agentic-Audio-RAG-Experiment


2025/08/28 04:22:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'AIStudio-Agentic-Audio-RAG-Model' already exists. Creating a new version of this model...
2025/08/28 04:22:40 WARNING mlflow.tracking._model_registry.fluent: Run with id bec15aa4bdae48cd9ecadc43aa54c55a has no artifacts at artifact path 'AIStudio-Agentic-Audio-RAG-Model', registering model based on models:/m-9cb73e5285c949e8b8faeaff1b067316 instead
Created version '11' of model 'AIStudio-Agentic-Audio-RAG-Model'.


Logged model at: runs:/bec15aa4bdae48cd9ecadc43aa54c55a/AIStudio-Agentic-Audio-RAG-Model
Registered name: AIStudio-Agentic-Audio-RAG-Model


CPU times: user 1.79 s, sys: 1.16 s, total: 2.95 s
Wall time: 16.9 s


In [13]:
loaded = mlflow.pyfunc.load_model(model_uri)

TEST_Q = "What is the main idea of the content?"
payload = [{"question": TEST_Q, "file_id": "global"}]

res = loaded.predict(payload)
print(json.dumps(res, indent=2)[:1200], "...")

✅ Loaded 1 secrets into environment variables.
CLAP moved to CPU; GPU cache cleared
Indexed segments: 18


You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

[
  {
    "question": "What is the main idea of the content?",
    "file_id": "global",
    "answer": "The! main!! idea! of!! the!\"content!is!that!!there!#!$!is!\"!!a!bug!%!&!!in!!the!!backend!$!$!related!to!creating!new!!projects!and!!the\"!model!dataset!!folder!not!'!(!!being!)!correct!!'.!It!!also!^!indicates!!that\"!!it!is!#!\"!\"!#!#!$!a!!user!related!issue!!rather!than!*!+!our!!own!bug!$.!If!you",
    "evidence": [
      {
        "file_name": "record for bug-20250512_113039-Meeting Recording.wav",
        "file_path": "/tmp/tmp58sbfdp8/AIStudio-Agentic-Audio-RAG-Model/data/model_artifacts/data/record for bug-20250512_113039-Meeting Recording.wav",
        "start_s": 30.0,
        "end_s": 60.0,
        "score": 0.24945804476737976
      },
      {
        "file_name": "record for bug-20250512_113039-Meeting Recording.mp4",
        "file_path": "/tmp/tmp58sbfdp8/AIStudio-Agentic-Audio-RAG-Model/data/model_artifacts/data/record for bug-20250512_113039-Meeting Recording.mp4",
    

# Load Model and Test Payload

In [13]:
loaded = mlflow.pyfunc.load_model(model_uri)

TEST_Q = "What is the main idea of the content?"
payload = [{"question": TEST_Q, "file_id": "global"}]

res = loaded.predict(payload)
print(json.dumps(res, indent=2)[:1200], "...")


✅ Loaded 1 secrets into environment variables.
CLAP moved to CPU; GPU cache cleared
Indexed segments: 18


preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

MlflowException: Failed to enforce schema of data '[{'question': 'What is the main idea of the content?', 'file_id': 'global'}]' with schema '['query': string (required), 'prompt': string (required), 'document': string (required)]'. Error: Model is missing inputs ['query', 'prompt', 'document']. Note that there were extra inputs: ['question', 'file_id']

# Message History

In [14]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).